# Student Performance Prediction

Predict a student's **final marks** from attendance, previous marks, assignment score, study hours, participation and other factors, and find which factors matter most.

**How to use:** click *Runtime -> Run all*. Everything runs from top to bottom with no files needed.

**Note:** the dataset is **created (synthetic)**, as the project brief allows ("select or create"). It is not real student data.

## 0. Create the dataset
Builds 1,000 students with some dirty data (missing values, duplicates, impossible values) on purpose, so the cleaning step has real work to do.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
N = 1000

# ---------- base student attributes ----------
gender = rng.choice(["Male", "Female"], N)
age = rng.integers(17, 23, N)
parent_education = rng.choice(
    ["High School", "Graduate", "Postgraduate"], N, p=[0.40, 0.42, 0.18]
)
extra_curricular = rng.choice(["Yes", "No"], N, p=[0.45, 0.55])
tutoring = rng.choice(["Yes", "No"], N, p=[0.30, 0.70])

# ---------- academic / behavioural factors ----------
study_hours = np.clip(rng.normal(12, 5, N), 0, 30)                 # per week
sleep_hours = np.clip(rng.normal(6.8, 1.0, N), 4, 9)               # per night
screen_time = np.clip(rng.normal(4.5, 1.8, N), 0.5, 10)            # mobile hrs/day

# previous marks depend a bit on study habits
previous_marks = np.clip(
    45 + 1.0 * study_hours + rng.normal(0, 11, N), 25, 98
)
# attendance depends on study habits and screen time
attendance = np.clip(
    78 + 0.4 * study_hours - 1.0 * screen_time + rng.normal(0, 9, N), 40, 100
)
# assignment score follows previous marks and study hours
assignment_score = np.clip(
    0.55 * previous_marks + 1.0 * study_hours + 15 + rng.normal(0, 8, N), 10, 100
)
participation = np.clip(
    
    np.round(5 + 0.04 * (attendance - 75) + 0.1 * (study_hours - 12) + rng.normal(0, 1.6, N)),
    1, 10,
)

# ---------- hidden formula for final marks ----------
final_marks = (
    0.38 * previous_marks
    + 0.22 * assignment_score
    + 0.14 * attendance
    + 0.55 * study_hours
    + 0.9 * participation
    + 0.6 * (sleep_hours - 6.8)
    - 0.9 * screen_time
    + 3.0 * (tutoring == "Yes")
    + 1.2 * (parent_education == "Postgraduate")
    + 0.5 * (parent_education == "Graduate")
    + rng.normal(0, 3.5, N)
    + 17
)
final_marks = np.clip(final_marks, 0, 100)

df = pd.DataFrame(
    {
        "student_id": [f"S{str(i).zfill(4)}" for i in range(1, N + 1)],
        "gender": gender,
        "age": age,
        "attendance_pct": attendance.round(1),
        "previous_marks": previous_marks.round(1),
        "assignment_score": assignment_score.round(1),
        "study_hours_per_week": study_hours.round(1),
        "participation_score": participation.astype(int),
        "sleep_hours": sleep_hours.round(1),
        "screen_time_hours": screen_time.round(1),
        "parent_education": parent_education,
        "extra_curricular": extra_curricular,
        "tutoring": tutoring,
        "final_marks": final_marks.round(1),
    }
)

# ---------- add realistic "dirty" data ----------
for col, frac in [
    ("attendance_pct", 0.03),
    ("assignment_score", 0.03),
    ("study_hours_per_week", 0.02),
    ("sleep_hours", 0.02),
    ("parent_education", 0.02),
]:
    idx = rng.choice(df.index, int(frac * N), replace=False)
    df.loc[idx, col] = np.nan

# a few impossible values (typing mistakes)
df.loc[rng.choice(df.index, 4, replace=False), "attendance_pct"] = 105
df.loc[rng.choice(df.index, 3, replace=False), "study_hours_per_week"] = 60

# a few duplicate rows
df = pd.concat([df, df.sample(8, random_state=1)], ignore_index=True)

import os
os.makedirs("data", exist_ok=True)
df.to_csv("data/student_performance_raw.csv", index=False)
print("Saved data/student_performance_raw.csv  shape =", df.shape)

## Setup
Imports, chart colours and helper functions.

In [ ]:
import os
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from sklearn.ensemble import (GradientBoostingRegressor, RandomForestClassifier,
                              RandomForestRegressor)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             f1_score, mean_absolute_error, mean_squared_error, r2_score)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
SEED = 42
os.makedirs("outputs", exist_ok=True)
os.makedirs("models", exist_ok=True)
LOG = []


def log(text=""):
    print(text)
    LOG.append(str(text))


# ---------- chart style (one blue main colour, orange for the second series) ----------
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.edgecolor": GRID, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlelocation": "left",
})


def save(fig, name):
    fig.tight_layout()
    fig.savefig(f"outputs/{name}", dpi=150)
    plt.show()
    plt.close(fig)

## 1. Load and inspect the data
Look at the raw data: how many rows, which values are missing, which are duplicated or impossible.

In [ ]:
# =====================================================================
# 1. LOAD + INSPECT
# =====================================================================
log("=" * 70)
log("1. LOAD AND INSPECT DATA")
log("=" * 70)
raw = pd.read_csv("data/student_performance_raw.csv")
log(f"Raw shape: {raw.shape[0]} rows x {raw.shape[1]} columns")
log("\nMissing values per column:")
log(raw.isna().sum()[raw.isna().sum() > 0].to_string())
log(f"\nDuplicate rows: {raw.duplicated().sum()}")
log(f"Attendance > 100%: {(raw.attendance_pct > 100).sum()} rows")
log(f"Study hours > 40/week: {(raw.study_hours_per_week > 40).sum()} rows")

## 2. Data cleaning and preprocessing
Remove duplicates, fix impossible values, fill missing values, drop the ID column.

In [ ]:
# =====================================================================
# 2. CLEAN + PREPROCESS
# =====================================================================
log("\n" + "=" * 70)
log("2. DATA CLEANING AND PREPROCESSING")
log("=" * 70)
df = raw.drop_duplicates().copy()
log(f"- Removed duplicates          -> {len(df)} rows left")

df.loc[df.attendance_pct > 100, "attendance_pct"] = np.nan
df.loc[df.study_hours_per_week > 40, "study_hours_per_week"] = np.nan
log("- Impossible values (attendance > 100, study hours > 40) set to missing")

num_cols = ["attendance_pct", "assignment_score", "study_hours_per_week", "sleep_hours"]
for c in num_cols:
    df[c] = df[c].fillna(df[c].median())
df["parent_education"] = df["parent_education"].fillna(df["parent_education"].mode()[0])
log("- Missing numbers filled with the median, missing categories with the most common value")
log(f"- Missing values left: {int(df.isna().sum().sum())}")

df = df.drop(columns=["student_id"])          # an ID has no predictive meaning
df.to_csv("data/student_performance_clean.csv", index=False)
log(f"- Dropped student_id. Clean data saved -> {df.shape[0]} rows x {df.shape[1]} columns")

log("\nSummary statistics (numeric columns):")
log(df.describe().round(1).T[["mean", "std", "min", "max"]].to_string())

## 3. Exploratory Data Analysis (EDA)
Charts that show how each factor relates to final marks.

In [ ]:
# =====================================================================
# 3. EDA
# =====================================================================
log("\n" + "=" * 70)
log("3. EXPLORATORY DATA ANALYSIS")
log("=" * 70)

# 3a. distribution of final marks
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df.final_marks, bins=25, color=BLUE, edgecolor=SURFACE, linewidth=1.5)
ax.axvline(df.final_marks.mean(), color=ORANGE, linewidth=2)
ax.text(df.final_marks.mean() + 0.8, ax.get_ylim()[1] * 0.93,
        f"Average = {df.final_marks.mean():.1f}", color=INK, fontsize=10)
ax.set_title("Most students score between 60 and 80")
ax.set_xlabel("Final marks")
ax.set_ylabel("Number of students")
save(fig, "eda_1_final_marks_distribution.png")

# 3b. correlation heatmap (diverging: orange = negative, blue = positive)
num_df = df.select_dtypes(include="number")
corr = num_df.corr()
cmap = LinearSegmentedColormap.from_list("div", [ORANGE, "#f1f0ec", BLUE])
fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(corr, cmap=cmap, vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)), corr.columns)
ax.grid(False)
for i in range(len(corr)):
    for j in range(len(corr)):
        v = corr.iloc[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(v) > 0.6 else INK)
ax.set_title("Correlation between numeric columns")
cb = fig.colorbar(im, ax=ax, fraction=0.04)
cb.set_label("Blue = move together, orange = move opposite")
save(fig, "eda_2_correlation_heatmap.png")

target_corr = corr["final_marks"].drop("final_marks").sort_values(ascending=False)
log("Correlation of each column with final_marks:")
log(target_corr.round(2).to_string())

# 3c. scatter plots for the 4 strongest numeric features
top4 = target_corr.abs().sort_values(ascending=False).index[:4]
fig, axes = plt.subplots(2, 2, figsize=(9, 7))
for ax, col in zip(axes.ravel(), top4):
    ax.scatter(df[col], df.final_marks, s=12, color=BLUE, alpha=0.4, linewidths=0)
    m, b = np.polyfit(df[col], df.final_marks, 1)
    xs = np.linspace(df[col].min(), df[col].max(), 50)
    ax.plot(xs, m * xs + b, color=ORANGE, linewidth=2)
    ax.set_xlabel(col)
    ax.set_ylabel("final_marks")
    ax.set_title(f"r = {target_corr[col]:.2f}", fontsize=10)
fig.suptitle("Strongest factors vs final marks (dots = students, line = trend)",
             x=0.02, ha="left", fontweight="bold")
save(fig, "eda_3_top_features_scatter.png")

# 3d. categorical columns vs average marks
cat_cols = ["gender", "parent_education", "extra_curricular", "tutoring"]
fig, axes = plt.subplots(1, 4, figsize=(12, 3.8), sharey=True)
for ax, col in zip(axes, cat_cols):
    means = df.groupby(col).final_marks.mean().sort_values()
    ax.bar(means.index, means.values, color=BLUE, width=0.6)
    for x, v in enumerate(means.values):
        ax.text(x, v + 0.6, f"{v:.1f}", ha="center", fontsize=9)
    ax.set_title(col, fontsize=10)
    ax.tick_params(axis="x", labelrotation=20)
    ax.set_ylim(0, 85)
axes[0].set_ylabel("Average final marks")
fig.suptitle("Average final marks by group: only tutoring makes a visible difference",
             x=0.01, ha="left", fontweight="bold")
save(fig, "eda_4_categorical_groups.png")

log("\nAverage final marks by group:")
for col in cat_cols:
    log(df.groupby(col).final_marks.mean().round(1).to_string())
    log("")

## 4. Train / test split
80% of students are used to teach the model, 20% are kept hidden to test it.

In [ ]:
# =====================================================================
# 4. TRAIN / TEST SPLIT
# =====================================================================
log("=" * 70)
log("4. TRAIN / TEST SPLIT")
log("=" * 70)
X = pd.get_dummies(df.drop(columns="final_marks"), drop_first=True)
y = df["final_marks"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
log(f"Features used ({X.shape[1]}): {list(X.columns)}")
log(f"Training set: {len(X_train)} students | Testing set: {len(X_test)} students (80/20 split)")

## 5-6. Train models and evaluate
Three models are trained and compared with MAE, RMSE and R2.

In [ ]:
# =====================================================================
# 5 + 6. TRAIN AND EVALUATE MODELS
# =====================================================================
log("\n" + "=" * 70)
log("5-6. TRAIN MODELS AND EVALUATE")
log("=" * 70)
models = {
    "Linear Regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=3,
                                           random_state=SEED, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=250, learning_rate=0.05,
                                                   max_depth=3, random_state=SEED),
}
rows, preds = [], {}
for name, model in models.items():
    cv = cross_val_score(model, X_train, y_train, cv=5, scoring="r2")
    model.fit(X_train, y_train)
    p = model.predict(X_test)
    preds[name] = p
    rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, p),
        "RMSE": np.sqrt(mean_squared_error(y_test, p)),
        "R2": r2_score(y_test, p),
        "CV R2 (5-fold, train)": cv.mean(),
    })
results = pd.DataFrame(rows).set_index("Model").round(3)
results.to_csv("outputs/model_comparison.csv")
log(results.to_string())
best_name = results["R2"].idxmax()
best_model = models[best_name]
log(f"\nBest model on the test set: {best_name} (R2 = {results.loc[best_name, 'R2']:.3f}, "
    f"MAE = {results.loc[best_name, 'MAE']:.2f} marks)")
baseline_mae = mean_absolute_error(y_test, np.full(len(y_test), y_train.mean()))
log(f"For comparison, always guessing the average gives MAE = {baseline_mae:.2f} marks")

# model comparison chart: error in marks (lower is better)
fig, ax = plt.subplots(figsize=(7, 3.6))
order = results.sort_values("MAE", ascending=False)
ax.barh(order.index, order["MAE"], color=BLUE, height=0.55)
for i, v in enumerate(order["MAE"]):
    ax.text(v + 0.05, i, f"{v:.2f}", va="center")
ax.set_xlabel("Average error on test students (marks) - lower is better")
ax.set_title("Model comparison")
ax.grid(axis="y", visible=False)
save(fig, "model_1_comparison.png")

# predicted vs actual + residuals for the best model
p = preds[best_name]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
axes[0].scatter(y_test, p, s=14, color=BLUE, alpha=0.5, linewidths=0)
lims = [min(y_test.min(), p.min()) - 2, max(y_test.max(), p.max()) + 2]
axes[0].plot(lims, lims, color=ORANGE, linewidth=2)
axes[0].set_xlim(lims); axes[0].set_ylim(lims)
axes[0].set_xlabel("Actual final marks")
axes[0].set_ylabel("Predicted final marks")
axes[0].set_title(f"{best_name}: predicted vs actual")
axes[0].text(lims[0] + 2, lims[1] - 6, "Orange line = perfect prediction", color=INK2)
resid = y_test - p
axes[1].hist(resid, bins=25, color=BLUE, edgecolor=SURFACE, linewidth=1.5)
axes[1].axvline(0, color=ORANGE, linewidth=2)
axes[1].set_xlabel("Error (actual - predicted)")
axes[1].set_ylabel("Number of students")
axes[1].set_title("Errors are centred around zero")
save(fig, "model_2_predicted_vs_actual.png")

## 7. Which factors matter most?
Permutation importance shows how much the score drops when a column is shuffled.

In [ ]:
# =====================================================================
# 7. IMPORTANT FEATURES
# =====================================================================
log("\n" + "=" * 70)
log("7. WHICH FACTORS MATTER MOST?")
log("=" * 70)
perm = permutation_importance(best_model, X_test, y_test, n_repeats=20,
                              random_state=SEED, scoring="r2")
imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
imp_std = pd.Series(perm.importances_std, index=X.columns)[imp.index]
log(f"Permutation importance from {best_name} (drop in R2 when a column is shuffled):")
log(imp.round(3).to_string())
imp.to_csv("outputs/feature_importance.csv", header=["importance"])

fig, ax = plt.subplots(figsize=(7.5, 5.4))
show = imp.sort_values()
ax.barh(show.index, show.values, xerr=imp_std[show.index].values, color=BLUE, height=0.6,
        error_kw={"ecolor": INK2, "elinewidth": 1})
ax.set_xlabel("Importance (drop in R2 when the column is shuffled)")
ax.set_title("What drives final marks?")
ax.grid(axis="y", visible=False)
save(fig, "model_3_feature_importance.png")

# linear regression coefficients (direction of effect, in marks per 1 std-dev)
lin = models["Linear Regression"]
coef = pd.Series(lin.named_steps["linearregression"].coef_, index=X.columns).sort_values()
log("\nLinear Regression: change in final marks for a +1 standard deviation increase in the column")
log(coef.sort_values(ascending=False).round(2).to_string())
fig, ax = plt.subplots(figsize=(7.5, 5.4))
colors = [BLUE if v > 0 else ORANGE for v in coef.values]
ax.barh(coef.index, coef.values, color=colors, height=0.6)
ax.axvline(0, color=INK2, linewidth=1)
ax.set_xlabel("Change in final marks per +1 std-dev  (blue = raises marks, orange = lowers)")
ax.set_title("Direction of each factor")
ax.grid(axis="y", visible=False)
save(fig, "model_4_direction_of_effect.png")

## 8. Extra: predict performance level
Classify students as *At Risk* (below 60), *Average* (60-74.9) or *Good* (75+).

In [ ]:
# =====================================================================
# 8. EXTRA - CLASSIFY STUDENTS INTO PERFORMANCE LEVELS
# =====================================================================
log("\n" + "=" * 70)
log("8. EXTRA: PREDICT PERFORMANCE LEVEL (At Risk / Average / Good)")
log("=" * 70)


def level(m):
    return "At Risk" if m < 60 else ("Average" if m < 75 else "Good")


yc = y.apply(level)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X, yc, test_size=0.2, random_state=SEED, stratify=yc)
clf = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=SEED,
                             n_jobs=-1, class_weight="balanced")
clf.fit(Xc_train, yc_train)
pc = clf.predict(Xc_test)
log("Levels: At Risk = below 60, Average = 60 to 74.9, Good = 75 and above")
log(f"Class counts:\n{yc.value_counts().to_string()}")
log(f"\nAccuracy: {accuracy_score(yc_test, pc):.3f} | Macro F1: {f1_score(yc_test, pc, average='macro'):.3f}")
log("\n" + classification_report(yc_test, pc))
fig, ax = plt.subplots(figsize=(5.5, 4.8))
ConfusionMatrixDisplay.from_predictions(
    yc_test, pc, labels=["At Risk", "Average", "Good"], ax=ax,
    cmap=LinearSegmentedColormap.from_list("seq", ["#f4f8fd", BLUE]), colorbar=False)
ax.grid(False)
ax.set_title("Performance level: predicted vs actual")
save(fig, "model_5_confusion_matrix.png")

## 9. Save the model and predict for a new student

In [ ]:
# =====================================================================
# 9. SAVE MODEL + PREDICT FOR A NEW STUDENT
# =====================================================================
log("\n" + "=" * 70)
log("9. SAVE MODEL AND PREDICT FOR A NEW STUDENT")
log("=" * 70)
joblib.dump({"model": best_model, "columns": list(X.columns), "name": best_name},
            "models/best_model.joblib")
log(f"Saved: models/best_model.joblib ({best_name})")

new = pd.DataFrame([{
    "gender": "Female", "age": 20, "attendance_pct": 85, "previous_marks": 72,
    "assignment_score": 75, "study_hours_per_week": 14, "participation_score": 7,
    "sleep_hours": 7, "screen_time_hours": 3.5, "parent_education": "Graduate",
    "extra_curricular": "Yes", "tutoring": "No",
}])
new_X = pd.get_dummies(new).reindex(columns=X.columns, fill_value=0)
pred = float(best_model.predict(new_X)[0])
log("\nExample new student: attendance 85%, previous marks 72, assignments 75, "
    "14 study hrs/week, participation 7/10")
log(f"Predicted final marks: {pred:.1f}  ->  level: {level(pred)}")

with open("outputs/results.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(LOG))
print("\nDone. Charts and results are in the 'outputs' folder.")

## 10. Download everything (optional)
Zips the data, charts and model so you can download them from Colab.

In [ ]:
import shutil
shutil.make_archive('student_project_outputs', 'zip', '.', 'outputs')
try:
    from google.colab import files
    files.download('student_project_outputs.zip')
except ImportError:
    print('Not running in Colab. Files are in the outputs/ folder.')